# IGP: QLoRA Fine-tuning — Qwen2.5 (0.5B / 1.5B) on Kaggle T4
**Karan — UWE Bristol MSc Data Science IGP**

### Before running:
1. Enable GPU: Settings > Accelerator > GPU T4 x2
2. Qwen2.5 is Apache 2.0, so no licence and no HuggingFace token are needed.
3. Add the sme-voice-data dataset as input (contains sme_train_v2.jsonl and sme_val_v2.jsonl)
4. Pick the size in Step 3 (0.5B by default; switch to 1.5B by changing MODEL_ID and OUTPUT_DIR).
5. The Step 9 sanity cell must print `has action: True` before you download the adapter.

In [ ]:
import os, torch
# Qwen2.5 is public (Apache 2.0), no HF token needed.
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# Step 2: Install packages
# This takes 2-3 minutes on first run.
# peft and accelerate are pinned (were >= before): an unpinned newer peft is the
# leading suspect for the v2 adapter dropping the action label. Pin to versions
# matched to transformers 4.45.2.
!pip install -q \
  "transformers==4.45.2" \
  "trl==0.11.4" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.5" \
  "accelerate==0.34.2" \
  datasets
import peft, accelerate, transformers, trl
print('versions:', 'transformers', transformers.__version__, '| trl', trl.__version__,
      '| peft', peft.__version__, '| accelerate', accelerate.__version__)

In [ ]:
# Step 3: Data + model config
TRAIN_FILE = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_train_v2.jsonl'
VAL_FILE   = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_val_v2.jsonl'

# 0.5B by default (fastest). For the 1.5B, swap both lines to:
#   MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
#   OUTPUT_DIR = '/kaggle/working/checkpoints/sme-qwen1.5b-qlora-v2'
MODEL_ID   = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = '/kaggle/working/checkpoints/sme-qwen0.5b-qlora-v2'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths set:', MODEL_ID)

In [ ]:
# Step 4: Load model + tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

# T4 = Turing, no flash_attention_2
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb,
    device_map='auto', torch_dtype=torch.bfloat16, attn_implementation='eager',
)
model.config.use_cache = False
print('Model loaded.')

In [ ]:
# Step 5: Apply QLoRA
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', inference_mode=False,
))
model.print_trainable_parameters()

In [ ]:
# Step 6: Load and format dataset (Qwen2.5 chat template)
from datasets import load_dataset

def fmt(rec):
    return {'text': (
        f"<|im_start|>system\n{rec['instruction']}<|im_end|>\n"
        f"<|im_start|>user\n{rec['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{rec['output']}<|im_end|>"
    )}

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train').map(fmt)
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train').map(fmt)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Step 7: Train
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

collator = DataCollatorForCompletionOnlyLM(
    response_template='<|im_start|>assistant\n',
    tokenizer=tokenizer,
)

trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=16,
        gradient_checkpointing=True,
        optim='paged_adamw_32bit',
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        bf16=True, tf32=False,
        max_grad_norm=0.3,
        report_to='none',
        dataset_text_field='text',
        max_seq_length=512,
    ),
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

print('Starting training... (Qwen2.5 0.5B/1.5B on T4, expect ~10-20 min)')
trainer.train()

In [ ]:
import json
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
with open(f'{OUTPUT_DIR}/training_meta.json','w') as f:
    json.dump({'model_id': MODEL_ID, 'lora_r': 16, 'lora_alpha': 32,
               'epochs': 3, 'lr': 2e-4, 'chat_template': 'qwen',
               'train_samples': len(train_ds)}, f, indent=2)
print(f'Done. Download from Output tab > {OUTPUT_DIR}')

In [ ]:
# Step 9: Sanity check - confirm the model emits the action label BEFORE you download it.
# If "has action: False" prints, the run is broken. Fix and retrain, do not waste a download.
model.config.use_cache = True
model.eval()
SYS = ("Appointment assistant. Output one JSON object only. "
       "Actions: book_appointment, check_availability, cancel_appointment, clarify, out_of_scope. "
       "Services: general, consultation, follow_up. Dates: YYYY-MM-DD. Times: HH:MM. "
       'If fields missing: {"action": "clarify", "missing_fields": [...]}.')
imend = tokenizer.convert_tokens_to_ids('<|im_end|>')
for t in ["Please book a consultation on Monday at 2pm.", "Do you have anything on Tuesday?"]:
    p = f"<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n{t}<|im_end|>\n<|im_start|>assistant\n"
    ids = tokenizer(p, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=60, do_sample=False,
                         eos_token_id=imend, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print('IN :', t); print('OUT:', txt); print('has action:', '"action"' in txt); print()